# Chapter 3 — Layers & the Forward Pass

**The building block this notebook untangles:** one neuron can only draw one
straight decision boundary (Chapter 1). A **layer** is just many neurons looking
at the *same* inputs in parallel, each with its own weights — and stacking
layers is what lets a network combine several straight boundaries into a bent
one. Running many neurons one-by-one in a Python loop would work but is slow and
ugly; instead, real code expresses a whole layer as **one matrix multiplication**.

## Part 1 — From one neuron to a layer

One neuron: `z = x·w + b`, a single number.

A layer of `k` neurons: stack `k` weight vectors into a matrix `W` of shape
`(n_inputs, k)`, and `k` biases into a vector `b`. Then:

$$Z = XW + b \qquad A = \text{activation}(Z)$$

`X` can hold many samples at once (one row per sample), so this same formula
processes an entire batch of data in a single matrix multiply.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

# 4 samples (the XOR truth table inputs), 2 features each
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)

# A layer of 3 neurons, each looking at both inputs
rng = np.random.default_rng(1)
W1 = rng.normal(scale=1.0, size=(2, 3))   # (n_inputs=2, n_neurons=3)
b1 = np.zeros(3)

Z1 = X @ W1 + b1        # (4, 3): one row per sample, one column per neuron
A1 = relu(Z1)

print("X shape:", X.shape, " W1 shape:", W1.shape, " -> Z1 shape:", Z1.shape)
print("\nZ1 (pre-activation):\n", np.round(Z1, 2))
print("\nA1 (after ReLU):\n", np.round(A1, 2))

X shape: (4, 2)  W1 shape: (2, 3)  -> Z1 shape: (4, 3)

Z1 (pre-activation):
 [[ 0.    0.    0.  ]
 [-1.3   0.91  0.45]
 [ 0.35  0.82  0.33]
 [-0.96  1.73  0.78]]

A1 (after ReLU):
 [[0.   0.   0.  ]
 [0.   0.91 0.45]
 [0.35 0.82 0.33]
 [0.   1.73 0.78]]


Read `A1` column-by-column: column 0 is "what neuron 1 thinks about every
sample," column 1 is neuron 2's opinion, and so on. Three neurons, one matrix
multiply, no loop.

## Part 2 — Stacking layers: the forward pass

A network is layers chained together, each one's output feeding the next one's
input. We'll build the network this whole series has been building toward:
**2 inputs → 4 hidden neurons (ReLU) → 1 output neuron (sigmoid)** — enough
capacity, eventually, to learn XOR.

In [2]:
n_in, n_hidden, n_out = 2, 4, 1

rng = np.random.default_rng(7)
W1 = rng.normal(scale=1.0, size=(n_in, n_hidden))
b1 = np.zeros(n_hidden)
W2 = rng.normal(scale=1.0, size=(n_hidden, n_out))
b2 = np.zeros(n_out)

def forward(X, W1, b1, W2, b2):
    """Two-layer forward pass. Returns every intermediate value —
    we'll need them again for backprop in Chapter 5."""
    Z1 = X @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)
    return Z1, A1, Z2, A2

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])   # XOR: 1 only when inputs differ

Z1, A1, Z2, A2 = forward(X_xor, W1, b1, W2, b2)

print(f"{'x1':>3} {'x2':>3} {'network output':>15} {'target (XOR)':>13}")
for x, out, target in zip(X_xor, A2.ravel(), y_xor):
    print(f"{x[0]:>3.0f} {x[1]:>3.0f} {out:>15.4f} {target:>13}")

 x1  x2  network output  target (XOR)
  0   0          0.5000             0
  0   1          0.6243             1
  1   0          0.4536             1
  1   1          0.5400             0


With **random** weights the outputs are noise — around `0.5`, no relation to the
target column. That's expected: nothing has *learned* anything yet. The forward
pass is only half the story; Chapters 4–6 build the other half (measuring the
error, and using it to fix the weights).

## Part 3 — The network is one function, end to end

Every layer's output shape must match the next layer's expected input shape —
that's the only "wiring" constraint. Print the shapes to see the chain:

In [3]:
for name, arr in [('X', X_xor), ('W1', W1), ('Z1/A1', Z1), ('W2', W2), ('Z2/A2', Z2)]:
    print(f"{name:>6}: {arr.shape}")

     X: (4, 2)
    W1: (2, 4)
 Z1/A1: (4, 4)
    W2: (4, 1)
 Z2/A2: (4, 1)


## Recap

A layer is a matrix multiply plus a bias plus an activation; a network is
several of those chained, each one's output shape feeding the next one's input
shape. We can now compute a prediction for *any* weights — Chapter 4 gives us a
way to score how good (or bad) that prediction is.

**Next:** [Chapter 4 — Loss Functions](04_loss_functions.ipynb)